# Bilibili 影片下載 + Markdown 轉換器 v3.1 (修復版)

使用 bilix 下載，Whisper 轉錄

流程：下載影片 → 儲存 Google Drive → 轉錄 → 生成 Markdown

## Step 1: 連接 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

WORK_DIR = "/content/drive/MyDrive/bilibili-projects"
VIDEO_DIR = f"{WORK_DIR}/videos"
AUDIO_DIR = f"{WORK_DIR}/audio"
OUTPUT_DIR = f"{WORK_DIR}/markdown"

os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

%cd {WORK_DIR}
print("✅ 已連接 Google Drive")

## Step 2: 安裝 bilix (Bilibili 下載器)

In [ ]:
!pip install -q bilix
print("✅ bilix 安裝完成")

## Step 3: 設定並下載影片

In [ ]:
# 設定
VIDEO_URL = "https://space.bilibili.com/1925268550/lists/839662?type=season"
QUALITY = 80  # 1080P
LIMIT = 3  # 先測試 3 條

print(f"URL: {VIDEO_URL}")
print(f"畫質: {QUALITY}P")
print(f"數量: {LIMIT}")

In [ ]:
import subprocess

# 使用 bilix 下載
cmd = [
    "bilix",
    "v",
    VIDEO_URL,
    "--dir", VIDEO_DIR,
    "--quality", str(QUALITY),
    "--num", str(LIMIT)
]

print("開始下載...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("錯誤:", result.stderr)

In [ ]:
# 檢查下載嘅檔案
import glob

mp4_files = glob.glob(f"{VIDEO_DIR}/**/*.mp4", recursive=True)
flv_files = glob.glob(f"{VIDEO_DIR}/**/*.flv", recursive=True)
mkv_files = glob.glob(f"{VIDEO_DIR}/**/*.mkv", recursive=True)

downloaded_videos = mp4_files + flv_files + mkv_files

print(f"找到 {len(downloaded_videos)} 個影片")
for i, v in enumerate(downloaded_videos, 1):
    size = os.path.getsize(v) / (1024*1024)
    print(f"{i}. {os.path.basename(v)} ({size:.1f} MB)")

## Step 4: 安裝 Whisper

In [ ]:
!pip install -q openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg
print("✅ Whisper 安裝完成")

## Step 5: 轉錄設定

In [ ]:
WHISPER_MODEL = "small"
LANGUAGE = "zh"

print(f"模型: {WHISPER_MODEL}")
print(f"語言: {LANGUAGE}")

## Step 6: 語音轉錄並生成 Markdown

In [ ]:
import whisper
from pathlib import Path
from datetime import datetime

model = whisper.load_model(WHISPER_MODEL)
print("✅ 模型載入完成")

In [ ]:
def extract_audio(video_path, audio_path):
    cmd = [
        "ffmpeg", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le",
        "-ar", "16000", "-ac", "1",
        "-y", audio_path
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    return audio_path

results = []

for i, video_path in enumerate(downloaded_videos, 1):
    name = Path(video_path).stem
    print(f"\n[{i}/{len(downloaded_videos)}] {name}")
    
    # 提取音軌
    audio_path = f"{AUDIO_DIR}/{name}.wav"
    extract_audio(video_path, audio_path)
    
    # 轉錄
    result = model.transcribe(audio_path, language=LANGUAGE)
    
    # 生成 Markdown
    md_path = f"{OUTPUT_DIR}/{name}.md"
    size_mb = os.path.getsize(video_path) / (1024*1024)
    
    content = f"# {name}\n\n"
    content += "## 元數據\n\n"
    content += f"- 檔案大小: {size_mb:.1f} MB\n"
    content += f"- 生成日期: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"
    content += "---\n\n## 內容轉錄\n\n"
    
    for seg in result['segments']:
        start = int(seg['start'] // 60), int(seg['start'] % 60)
        end = int(seg['end'] // 60), int(seg['end'] % 60)
        content += f"### {start[0]:02d}:{start[1]:02d} - {end[0]:02d}:{end[1]:02d}\n\n"
        content += f"{seg['text'].strip()}\n\n"
    
    content += "---\n\n*由 bilibili-to-markdown 生成*\n"
    
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(content)
    
    results.append({'title': name, 'md': md_path})
    print(f"✅ 完成: {name}.md")

print(f"\n📊 完成: {len(results)} 個影片")

## Step 7: 生成索引

In [ ]:
summary = f"{OUTPUT_DIR}/summary.md"
with open(summary, 'w', encoding='utf-8') as f:
    f.write("# Bilibili 轉錄索引\n\n")
    f.write(f"總數: {len(results)}\n\n")
    for i, r in enumerate(results, 1):
        f.write(f"{i}. [{r['title']}](./{Path(r['md']).name})\n")
print(f"✅ 索引: {summary}")

## ✅ 完成！

In [ ]:
print("\n🎉 全部完成！")
print(f"📁 影片: {VIDEO_DIR}")
print(f"📝 Markdown: {OUTPUT_DIR}")